# 🏠 Cost Segregation Tax Worksheet
### Depreciation Deduction Report — IRC §168 MACRS with 40% Bonus Depreciation

This notebook calculates a full cost segregation study for a residential rental property  
and produces a professionally formatted PDF report using **ReportLab**.

---
**Sections:**
1. Install & Import Dependencies
2. Property Inputs
3. Cost Segregation Calculations
4. Bonus Depreciation Schedule
5. Year-1 Deduction Summary
6. Benefit Analysis
7. Generate PDF Report

---
## Section 1 — Install & Import Dependencies

In [19]:
# 1. Install reportlab if not already present
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab", "--quiet"])
print("✅ reportlab ready")

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
print("✅ All imports successful")

✅ reportlab ready
✅ All imports successful


## Section 2 - Property InputsBasis

In [ ]:
# ─ 2 ─ PROPERTY INPUTS ─────────────────────────────────────────
purchase_price = 220_000.00   # Total acquisition cost
land_value     =  63_720.00   # Non-depreciable land portion
improvements   =  10_000.00   # Capital improvements added to basis
bonus_rate     =       0.40   # Bonus depreciation rate (40% per TCJA phase-down)

In [ ]:
# ─ 3 ─ COST SEGREGATION ALLOCATION PERCENTAGES ─────────────────
# IRS-typical splits for residential rental property
residential_pct   = 0.75   # 27.5-yr straight-line (§1250 building)
land_improve_pct  = 0.10   # 15-yr MACRS (land improvements)
personal_prop_pct = 0.15   # 5-yr MACRS 200DB (§1245 personal property)

print(f"Purchase Price : ${purchase_price:>12,.2f}")
print(f"Land Value     : ${land_value:>12,.2f}")
print(f"Improvements   : ${improvements:>12,.2f}")
print(f"Bonus Rate     : {bonus_rate*100:.0f}%")

Purchase Price : $  220,000.00
Land Value     : $   63,720.00
Improvements   : $   10,000.00
Bonus Rate     : 40%


---
## Section 3 — Cost Segregation Calculations

In [29]:
# - 4a -- Depreciable Basis, Asset Class Allocation, Recovery Periods
# ── STEP 1: DEPRECIABLE BASIS ────────────────────────────────
building_basis_raw = purchase_price - land_value
depreciable_basis  = building_basis_raw + improvements

# ── STEP 2: ASSET CLASS ALLOCATIONS ─────────────────────────
residential_amount   = depreciable_basis * residential_pct
land_improve_amount  = depreciable_basis * land_improve_pct
personal_prop_amount = depreciable_basis * personal_prop_pct

# ── STEP 3: RECOVERY PERIODS ────────────────────────────────
residential_life   = 27.5
land_improve_life  = 15.0
personal_prop_life =  5.0

print("=" * 48)
print(f"  Building Basis (raw)         : ${building_basis_raw:>11,.2f}")
print(f"  Depreciable Basis            : ${depreciable_basis:>11,.2f}")
print("-" * 48)
print(f"  Residential (75%, 27.5yr).   : ${residential_amount:>11,.2f}")
print(f"  Land Improvements (10%, 15yr): ${land_improve_amount:>11,.2f}")
print(f"  Personal Property (15%, 5yr) : ${personal_prop_amount:>11,.2f}")
print(f"  CHECK — Sum.                 : ${residential_amount+land_improve_amount+personal_prop_amount:>11,.2f}")
print("=" * 48)

  Building Basis (raw)         : $ 156,280.00
  Depreciable Basis            : $ 166,280.00
------------------------------------------------
  Residential (75%, 27.5yr).   : $ 124,710.00
  Land Improvements (10%, 15yr): $  16,628.00
  Personal Property (15%, 5yr) : $  24,942.00
  CHECK — Sum.                 : $ 166,280.00


---
## Section 4 — Bonus Depreciation Schedule (40%)

In [23]:
# ─ 4b ─ BONUS DEPRECIATION ──────────────────────────────────────
# 27.5-yr residential structure is NOT eligible under IRC §168(k)
# Only 5-yr personal property and 15-yr land improvements qualify

bonus_personal  = personal_prop_amount * bonus_rate
bonus_land_impr = land_improve_amount  * bonus_rate
total_bonus     = bonus_personal + bonus_land_impr

# ── REMAINING BASIS AFTER BONUS ─────────────────────────────
remaining_personal  = personal_prop_amount - bonus_personal
remaining_land_impr = land_improve_amount  - bonus_land_impr

print(f"Bonus — Personal Property (40%): ${bonus_personal:>10,.2f}")
print(f"Bonus — Land Improvements (40%): ${bonus_land_impr:>10,.2f}")
print(f"TOTAL BONUS DEPRECIATION       : ${total_bonus:>10,.2f}")
print()
print(f"Remaining Personal Prop basis  : ${remaining_personal:>10,.2f}")
print(f"Remaining Land Impr basis      : ${remaining_land_impr:>10,.2f}")

Bonus — Personal Property (40%): $  9,976.80
Bonus — Land Improvements (40%): $  6,651.20
TOTAL BONUS DEPRECIATION       : $ 16,628.00

Remaining Personal Prop basis  : $ 14,965.20
Remaining Land Impr basis      : $  9,976.80


---
## Section 5 — Year 1 Total Depreciation Deduction

In [13]:
# ─ 5 ─ 1 yr Deduction summary / REGULAR MACRS ON REMAINING BALANCES ─────────────────────
# Residential: SL full-year
yr1_regular_residential = residential_amount / residential_life

# Land Improvements: SL on remaining basis (post-bonus)
yr1_regular_land_impr   = remaining_land_impr / land_improve_life

# Personal Property: 200DB with half-year convention on remaining basis
yr1_regular_personal    = remaining_personal * (2 / personal_prop_life) * 0.5

yr1_total_regular = (yr1_regular_residential
                   + yr1_regular_land_impr
                   + yr1_regular_personal)

yr1_grand_total   = total_bonus + yr1_total_regular

# ── COMPARISON BASELINE ─────────────────────────────────────
straight_line_annual  = depreciable_basis / residential_life
yr1_advantage         = yr1_grand_total - straight_line_annual

print("YEAR 1 DEPRECIATION BREAKDOWN")
print("=" * 50)
print(f"A. Residential SL (27.5yr)      : ${yr1_regular_residential:>10,.2f}")
print(f"B. Land Impr MACRS (post-bonus) : ${yr1_regular_land_impr:>10,.2f}")
print(f"C. Personal Prop MACRS (post-bon): ${yr1_regular_personal:>9,.2f}")
print(f"D. Bonus — Land Improvements    : ${bonus_land_impr:>10,.2f}")
print(f"E. Bonus — Personal Property    : ${bonus_personal:>10,.2f}")
print("-" * 50)
print(f"   YEAR 1 TOTAL DEDUCTION       : ${yr1_grand_total:>10,.2f}")
print()
print(f"   Traditional SL (no cost seg) : ${straight_line_annual:>10,.2f}")
print(f"   YEAR 1 ADVANTAGE             : ${yr1_advantage:>10,.2f}")
print("=" * 50)

YEAR 1 DEPRECIATION BREAKDOWN
A. Residential SL (27.5yr)      : $  4,534.91
B. Land Impr MACRS (post-bonus) : $    665.12
C. Personal Prop MACRS (post-bon): $ 2,993.04
D. Bonus — Land Improvements    : $  6,651.20
E. Bonus — Personal Property    : $  9,976.80
--------------------------------------------------
   YEAR 1 TOTAL DEDUCTION       : $ 24,821.07

   Traditional SL (no cost seg) : $  6,046.55
   YEAR 1 ADVANTAGE             : $ 18,774.52


---
## Section 6 — Benefit Analysis

In [21]:
# - 6 - Benefits Analysis (tax rate, NPV discount)
tax_rate   = 0.35   # Assumed combined federal + state marginal rate
discount   = 0.05   # NPV discount rate

trad_tax_savings = straight_line_annual * tax_rate
costseg_tax_savings = yr1_grand_total   * tax_rate
yr1_extra_savings   = yr1_advantage     * tax_rate
npv_benefit         = yr1_extra_savings / (1 + discount)

print("BENEFIT ANALYSIS (@ 35% tax rate, 5% NPV discount)")
print("=" * 52)
print(f"Traditional SL tax savings (Yr 1) : ${trad_tax_savings:>9,.2f}")
print(f"Cost Seg tax savings (Yr 1)        : ${costseg_tax_savings:>9,.2f}")
print(f"Extra Year-1 tax savings            : ${yr1_extra_savings:>9,.2f}")
print(f"NPV of tax deferral benefit         : ${npv_benefit:>9,.2f}")
print("=" * 52)

BENEFIT ANALYSIS (@ 35% tax rate, 5% NPV discount)
Traditional SL tax savings (Yr 1) : $ 2,116.29
Cost Seg tax savings (Yr 1)        : $ 8,687.37
Extra Year-1 tax savings            : $ 6,571.08
NPV of tax deferral benefit         : $ 6,258.17


---
## Section 7 — Generate PDF Report
Runs ReportLab to produce the full formatted PDF worksheet.

In [15]:
# - 7 - Generate Report
# ── COLOUR PALETTE ──────────────────────────────────────────
DARK_BLUE  = colors.HexColor("#1a3a5c")
MED_BLUE   = colors.HexColor("#2e6da4")
LIGHT_BLUE = colors.HexColor("#d6e8f7")
ACCENT     = colors.HexColor("#e8a020")
WHITE      = colors.white
GRAY_BG    = colors.HexColor("#f4f6f9")
BORDER     = colors.HexColor("#c8d4e0")
GREEN      = colors.HexColor("#1e7e34")

styles = getSampleStyleSheet()
W = 7.0 * inch

def style(name, **kw):
    parent = kw.pop("parent", styles["Normal"])
    return ParagraphStyle(name, parent=parent, **kw)

# paragraph styles
title_style   = style("Title",   fontSize=18, textColor=WHITE,     alignment=TA_CENTER, fontName="Helvetica-Bold", spaceAfter=2)
sub_style     = style("Sub",     fontSize=10, textColor=LIGHT_BLUE, alignment=TA_CENTER, fontName="Helvetica",      spaceAfter=2)
h1_style      = style("H1",      fontSize=12, textColor=WHITE,     fontName="Helvetica-Bold", leftIndent=6)
note_style    = style("Note",    fontSize=8,  textColor=colors.HexColor("#555555"), fontName="Helvetica-Oblique")
label_style   = style("Label",   fontSize=9,  textColor=DARK_BLUE, fontName="Helvetica")
value_style   = style("Value",   fontSize=9,  textColor=DARK_BLUE, fontName="Helvetica-Bold", alignment=TA_RIGHT)
total_lbl     = style("TotLbl",  fontSize=9,  textColor=WHITE,     fontName="Helvetica-Bold")
total_val     = style("TotVal",  fontSize=9,  textColor=WHITE,     fontName="Helvetica-Bold", alignment=TA_RIGHT)
footer_style  = style("Footer",  fontSize=7.5,textColor=colors.HexColor("#888888"), alignment=TA_CENTER)

def fmt(n): return f"${n:,.2f}"

def section_header(title):
    t = Table([[Paragraph(title, h1_style)]], colWidths=[W])
    t.setStyle(TableStyle([
        ("BACKGROUND",   (0,0),(-1,-1), MED_BLUE),
        ("TOPPADDING",   (0,0),(-1,-1), 5),
        ("BOTTOMPADDING",(0,0),(-1,-1), 5),
        ("LEFTPADDING",  (0,0),(-1,-1), 8),
    ]))
    return t

def cell(txt, align=TA_LEFT, bold=False, color=DARK_BLUE, size=8.5):
    fn = "Helvetica-Bold" if bold else "Helvetica"
    return Paragraph(txt, style(f"c_{txt[:6]}", fontSize=size, textColor=color, fontName=fn, alignment=align))

print("✅ Helpers defined")

✅ Helpers defined


In [31]:
# Generate File => Cost_Segregation_Tax_Worksheet.pdf
output_path = "Cost_Segregation_Tax_Worksheet.pdf"
doc   = SimpleDocTemplate(output_path, pagesize=letter,
                           leftMargin=0.75*inch, rightMargin=0.75*inch,
                           topMargin=0.75*inch,  bottomMargin=0.75*inch)
story = []

# ── HEADER BANNER ───────────────────────────────────────────
hdr = Table([
    [Paragraph("COST SEGREGATION STUDY", title_style)],
    [Paragraph("Depreciation Deduction Worksheet &nbsp;|&nbsp; IRC §168 MACRS with Bonus Depreciation (40%)", sub_style)],
], colWidths=[W])
hdr.setStyle(TableStyle([
    ("BACKGROUND",   (0,0),(-1,-1), DARK_BLUE),
    ("TOPPADDING",   (0,0),(-1, 0), 12),
    ("BOTTOMPADDING",(0,1),(-1, 1), 10),
    ("LEFTPADDING",  (0,0),(-1,-1), 10),
    ("RIGHTPADDING", (0,0),(-1,-1), 10),
    ("LINEBELOW",    (0,0),(-1, 0), 1, ACCENT),
]))
story += [hdr, Spacer(1, 10)]

# ── SECTION 1: PROPERTY INFORMATION ─────────────────────────
story.append(section_header("SECTION 1 — PROPERTY INFORMATION"))
story.append(Spacer(1, 2))
prop_data = [
    [Paragraph("Property Description",   label_style), Paragraph("Residential Rental Property", value_style)],
    [Paragraph("Total Purchase Price",   label_style), Paragraph(fmt(purchase_price),            value_style)],
    [Paragraph("Land Value (non-depr.)", label_style), Paragraph(fmt(land_value),                value_style)],
    [Paragraph("Capital Improvements",   label_style), Paragraph(fmt(improvements),              value_style)],
    [Paragraph("Building Basis (raw)",   label_style), Paragraph(fmt(building_basis_raw),        value_style)],
    [Paragraph("TOTAL DEPRECIABLE BASIS",total_lbl),   Paragraph(fmt(depreciable_basis),          total_val)],
]
pt = Table(prop_data, colWidths=[W*0.60, W*0.40])
pt.setStyle(TableStyle([
    ("ROWBACKGROUNDS",(0,0),(-1,4),[WHITE, GRAY_BG]),
    ("BACKGROUND",    (0,5),(-1, 5), DARK_BLUE),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (1,0),(1,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
    ("LINEABOVE",     (0,5),(-1, 5), 1.5, ACCENT),
]))
story += [pt, Spacer(1, 10)]

# ── SECTION 2: ASSET ALLOCATIONS ────────────────────────────
story.append(section_header("SECTION 2 — COST SEGREGATION ASSET ALLOCATIONS"))
story.append(Spacer(1, 2))
ah = [cell("Asset Class",bold=True,color=WHITE), cell("Recovery\nPeriod",TA_CENTER,True,WHITE),
      cell("Method",TA_CENTER,True,WHITE), cell("% Allocated",TA_RIGHT,True,WHITE), cell("Allocated\nAmount",TA_RIGHT,True,WHITE)]
alloc_data = [
    ah,
    [cell("Residential Building Structure (§1250)"), cell("27.5 yr",TA_CENTER), cell("SL",TA_CENTER),    cell("75.00%",TA_RIGHT), cell(fmt(residential_amount),  TA_RIGHT)],
    [cell("Land Improvements (§1250/§1245)"),        cell("15 yr",  TA_CENTER), cell("SL",TA_CENTER),    cell("10.00%",TA_RIGHT), cell(fmt(land_improve_amount), TA_RIGHT)],
    [cell("Personal Property — 5-yr (§1245)"),       cell("5 yr",   TA_CENTER), cell("200DB",TA_CENTER), cell("15.00%",TA_RIGHT), cell(fmt(personal_prop_amount),TA_RIGHT)],
    [cell("TOTAL",bold=True), cell("",TA_CENTER), cell("",TA_CENTER), cell("100.00%",TA_RIGHT,True), cell(fmt(depreciable_basis),TA_RIGHT,True)],
]
at = Table(alloc_data, colWidths=[W*0.36, W*0.12, W*0.12, W*0.14, W*0.26])
at.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1, 0), DARK_BLUE),
    ("ROWBACKGROUNDS",(0,1),(-1, 3),[WHITE, GRAY_BG, WHITE]),
    ("BACKGROUND",    (0,4),(-1, 4), LIGHT_BLUE),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (4,0),(4,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
    ("LINEABOVE",     (0,4),(-1, 4), 1.5, ACCENT),
]))
story += [at, Spacer(1, 10)]

# ── SECTION 3: BONUS DEPRECIATION ───────────────────────────
story.append(section_header("SECTION 3 — BONUS DEPRECIATION SCHEDULE (40% Rate per IRC §168(k))"))
story.append(Spacer(1, 2))
bh = [cell("Asset Class",bold=True,color=WHITE), cell("Eligible\nBasis",TA_RIGHT,True,WHITE),
      cell("Bonus\nRate",TA_CENTER,True,WHITE), cell("Bonus Deduction\nYear 1",TA_RIGHT,True,WHITE), cell("Remaining\nBasis",TA_RIGHT,True,WHITE)]
bonus_data = [
    bh,
    [cell("Residential Building (27.5 yr)"), cell(fmt(residential_amount),  TA_RIGHT), cell("N/A",TA_CENTER),  cell("—",TA_RIGHT),                           cell(fmt(residential_amount),  TA_RIGHT)],
    [cell("Land Improvements (15 yr)"),       cell(fmt(land_improve_amount), TA_RIGHT), cell("40%",TA_CENTER),  cell(fmt(bonus_land_impr),TA_RIGHT,False,GREEN), cell(fmt(remaining_land_impr), TA_RIGHT)],
    [cell("Personal Property — 5 yr"),        cell(fmt(personal_prop_amount),TA_RIGHT), cell("40%",TA_CENTER),  cell(fmt(bonus_personal), TA_RIGHT,False,GREEN), cell(fmt(remaining_personal),  TA_RIGHT)],
    [cell("TOTAL BONUS DEPRECIATION",bold=True), cell("",TA_RIGHT), cell("",TA_CENTER), cell(fmt(total_bonus),TA_RIGHT,True,GREEN), cell("",TA_RIGHT)],
]
bt = Table(bonus_data, colWidths=[W*0.35, W*0.17, W*0.10, W*0.20, W*0.18])
bt.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1, 0), DARK_BLUE),
    ("ROWBACKGROUNDS",(0,1),(-1, 3),[WHITE, GRAY_BG, WHITE]),
    ("BACKGROUND",    (0,4),(-1, 4), LIGHT_BLUE),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (4,0),(4,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
    ("LINEABOVE",     (0,4),(-1, 4), 1.5, ACCENT),
]))
story += [bt, Spacer(1,4),
          Paragraph("\u2605  27.5-year residential building structure is NOT eligible for bonus depreciation "
                    "under IRC \u00a7168(k). Only \u00a71245 personal property and land improvements qualify.", note_style),
          Spacer(1, 10)]

# ── SECTION 4: YEAR-1 SUMMARY ───────────────────────────────
story.append(section_header("SECTION 4 — YEAR 1 TOTAL DEPRECIATION DEDUCTION SUMMARY"))
story.append(Spacer(1, 2))
sh = [cell("Deduction Component",bold=True,color=WHITE), cell("Calculation Basis",TA_CENTER,True,WHITE), cell("Year 1 Amount",TA_RIGHT,True,WHITE)]
sum_data = [
    sh,
    [cell(f"A.  Residential Structure — Regular MACRS (SL)"),         cell(f"{fmt(residential_amount)} ÷ 27.5 yrs",TA_CENTER),                         cell(fmt(yr1_regular_residential),TA_RIGHT)],
    [cell(f"B.  Land Improvements — Regular MACRS (post-bonus)"),      cell(f"{fmt(remaining_land_impr)} ÷ 15 yrs",TA_CENTER),                         cell(fmt(yr1_regular_land_impr),  TA_RIGHT)],
    [cell(f"C.  Personal Property — Regular MACRS (post-bonus)"),      cell(f"{fmt(remaining_personal)} × 40% × ½",TA_CENTER),                        cell(fmt(yr1_regular_personal),   TA_RIGHT)],
    [cell(f"D.  Bonus Depreciation — Land Improvements (40%)"),        cell(f"{fmt(land_improve_amount)} × 40%",TA_CENTER),                            cell(fmt(bonus_land_impr),        TA_RIGHT,False,GREEN)],
    [cell(f"E.  Bonus Depreciation — Personal Property (40%)"),        cell(f"{fmt(personal_prop_amount)} × 40%",TA_CENTER),                           cell(fmt(bonus_personal),         TA_RIGHT,False,GREEN)],
    [cell("YEAR 1 TOTAL DEPRECIATION DEDUCTION",bold=True,color=WHITE),cell("Lines A + B + C + D + E",TA_CENTER,True,WHITE),                           cell(fmt(yr1_grand_total),        TA_RIGHT,True,GREEN)],
]
st = Table(sum_data, colWidths=[W*0.45, W*0.32, W*0.23])
st.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1, 0), DARK_BLUE),
    ("ROWBACKGROUNDS",(0,1),(-1, 5),[WHITE, GRAY_BG, WHITE, GRAY_BG, WHITE]),
    ("BACKGROUND",    (0,6),(-1, 6), DARK_BLUE),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (2,0),(2,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
    ("LINEABOVE",     (0,6),(-1, 6), 2,   ACCENT),
]))
story += [st, Spacer(1, 10)]

# ── SECTION 5: BENEFIT ANALYSIS ─────────────────────────────
story.append(section_header("SECTION 5 — BENEFIT ANALYSIS: COST SEG vs. TRADITIONAL STRAIGHT-LINE"))
story.append(Spacer(1, 2))
bah = [cell("Metric",bold=True,color=WHITE),
       cell("Traditional SL\n(27.5 yr only)",TA_CENTER,True,WHITE),
       cell("Cost Segregation\n(with 40% Bonus)",TA_CENTER,True,WHITE),
       cell("Year 1 Advantage",TA_RIGHT,True,WHITE)]
ben_data = [
    bah,
    [cell("Depreciable Basis"),                         cell(fmt(depreciable_basis),      TA_CENTER), cell(fmt(depreciable_basis),      TA_CENTER),         cell("—",TA_RIGHT)],
    [cell("Year 1 Depreciation Deduction"),             cell(fmt(straight_line_annual),   TA_CENTER), cell(fmt(yr1_grand_total),TA_CENTER,False,GREEN),      cell(fmt(yr1_advantage),       TA_RIGHT,True,GREEN)],
    [cell("Assumed Tax Rate (combined 35%)"),           cell(fmt(trad_tax_savings),        TA_CENTER), cell(fmt(costseg_tax_savings),TA_CENTER,False,GREEN),  cell(fmt(yr1_extra_savings),   TA_RIGHT,True,GREEN)],
    [cell("NPV of Tax Deferral Benefit (5% discount)",bold=True), cell("Baseline",TA_CENTER,True), cell(fmt(npv_benefit),TA_CENTER,True,GREEN), cell(fmt(npv_benefit),TA_RIGHT,True,GREEN)],
]
bnt = Table(ben_data, colWidths=[W*0.36, W*0.20, W*0.24, W*0.20])
bnt.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1, 0), DARK_BLUE),
    ("ROWBACKGROUNDS",(0,1),(-1, 3),[WHITE, GRAY_BG, WHITE]),
    ("BACKGROUND",    (0,4),(-1, 4), LIGHT_BLUE),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (3,0),(3,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
    ("LINEABOVE",     (0,4),(-1, 4), 1.5, ACCENT),
]))
story += [bnt, Spacer(1, 10)]

# ── SECTION 6: IRS FORM REFERENCES ──────────────────────────
story.append(section_header("SECTION 6 — IRS FORM REFERENCES & REPORTING"))
story.append(Spacer(1, 4))
fh = [cell("IRS Form / Schedule",bold=True,color=WHITE), cell("Purpose",bold=True,color=WHITE), cell("Key Lines / Data Entry",bold=True,color=WHITE)]
form_data = [
    fh,
    [cell("Form 4562"),              cell("Depreciation & Amortization"),    cell("Part II (MACRS); Part VI (Bonus §168(k))")],
    [cell("Schedule E (Form 1040)"), cell("Supplemental Income & Loss"),     cell("Line 18 — Depreciation expense")],
    [cell("Form 8582"),              cell("Passive Activity Loss Limits"),    cell("Enter if rental losses exceed $25,000")],
    [cell("Asset Detail Record"),    cell("Cost Seg Study Support Doc"),      cell("Attach engineer's report; retain 7 years")],
]
ft = Table(form_data, colWidths=[W*0.25, W*0.35, W*0.40])
ft.setStyle(TableStyle([
    ("BACKGROUND",    (0,0),(-1, 0), DARK_BLUE),
    ("ROWBACKGROUNDS",(0,1),(-1,-1),[WHITE, GRAY_BG, WHITE, GRAY_BG]),
    ("TOPPADDING",    (0,0),(-1,-1), 5), ("BOTTOMPADDING",(0,0),(-1,-1), 5),
    ("LEFTPADDING",   (0,0),(0,-1),  8), ("RIGHTPADDING", (2,0),(2,-1),  8),
    ("GRID",          (0,0),(-1,-1), 0.4, BORDER),
]))
story += [ft, Spacer(1, 12)]

# ── FOOTER ──────────────────────────────────────────────────
story.append(HRFlowable(width=W, thickness=0.5, color=BORDER))
story.append(Spacer(1, 4))
story.append(Paragraph(
    "DISCLAIMER: This worksheet is prepared for informational and planning purposes only. It does not constitute "
    "legal or tax advice. All depreciation calculations are based on MACRS as prescribed under IRC §168 and the "
    "Tax Cuts and Jobs Act (TCJA). Bonus depreciation rates and eligibility are subject to change. Consult a "
    "licensed CPA or tax attorney before filing. 40% bonus depreciation rate applied consistent with TCJA phase-down schedule.",
    footer_style))

# ── BUILD ────────────────────────────────────────────────────
doc.build(story)
print(f"\n✅ PDF saved to: {output_path}")


✅ PDF saved to: Cost_Segregation_Tax_Worksheet.pdf


In [17]:
# ── DISPLAY INLINE (works in JupyterLab / classic Notebook) ─
import base64, pathlib
from IPython.display import display, HTML

pdf_bytes = pathlib.Path(output_path).read_bytes()
b64 = base64.b64encode(pdf_bytes).decode()
display(HTML(f"""
<h3>&#128196; Cost Segregation Tax Worksheet</h3>
<p>Click below to open / download the generated PDF.</p>
<a href=\"data:application/pdf;base64,{b64}\" 
   download=\"Cost_Segregation_Tax_Worksheet.pdf\"
   style=\"background:#1a3a5c;color:white;padding:8px 16px;border-radius:4px;
          text-decoration:none;font-family:sans-serif;font-size:14px;\"
>&#11015; Download PDF</a>
<br><br>
<iframe src=\"data:application/pdf;base64,{b64}\" width=\"900\" height=\"700\" 
        style=\"border:1px solid #ccc;border-radius:4px;\"></iframe>
"""))

### ⚠️ Disclaimer

This worksheet is for **informational and tax planning purposes only**. It does not constitute legal or tax advice.  
All figures are based on IRC §168 MACRS and the TCJA bonus depreciation phase-down schedule.  
Consult a licensed **CPA or tax attorney** before filing.  

**IRS References:** IRC §168 | Rev. Proc. 87-56 | TCJA §13201 | Form 4562 Instructions